# Symptom Checker - Real Excel Dataset

This notebook uses `../data/symptom_checker_dataset.xlsx` instead of generated sample rows. It performs cleaning, imputations/normalisation for messy inputs, EDA, visualisations, repeated fine tuning, neural-network-style trials, and saves the joblib model bundle used by the Flask backend.

The production model remains compatible with the current symptom checklist UI. The EDA still explores demographics and vitals from the real workbook.

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sys.path.insert(0, str(Path('..', 'scripts').resolve()))
from train_symptom import DISEASE_INFO, TARGET, clean_symptom_data

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'symptom_checker_dataset.xlsx'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

## 1. Load the real Excel workbook

In [ ]:
df_raw = pd.read_excel(DATA_PATH, sheet_name='SymptomChecker')
print('Raw shape:', df_raw.shape)
display(df_raw.head())
missing = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing': df_raw.isna().sum(),
    'missing_pct': (df_raw.isna().mean() * 100).round(2),
}).sort_values('missing_pct', ascending=False)
display(missing.head(15))

## 2. Clean and prepare

The imported cleaner trims text categories, keeps COVID-19 as a stable label, removes duplicates, checks plausible ages/vitals, and coerces symptom fields to binary 0/1. Missing symptom cells are filled with 0 because the checklist model treats unselected symptoms as absent.

In [ ]:
df, sx_cols = clean_symptom_data(df_raw)
symptoms = [c.replace('sx_', '') for c in sx_cols]
print('Clean shape:', df.shape)
print('Symptom count:', len(symptoms))
display(df[sx_cols + [TARGET]].head())

## 3. EDA and visualisations

In [ ]:
class_counts = df[TARGET].value_counts()
display(class_counts.to_frame('rows'))
plt.figure(figsize=(9, 6))
sns.barplot(x=class_counts.values, y=class_counts.index, color='#59A14F')
plt.title('Symptom diagnosis class balance')
plt.xlabel('Rows')
plt.ylabel('Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
prevalence = df.groupby(TARGET)[sx_cols].mean()
plt.figure(figsize=(16, 8))
sns.heatmap(prevalence, cmap='YlGnBu', cbar_kws={'label': 'P(symptom | diagnosis)'})
plt.title('Symptom prevalence by diagnosis')
plt.xlabel('Symptom')
plt.ylabel('Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
symptom_frequency = df[sx_cols].sum().sort_values()
plt.figure(figsize=(8, 9))
symptom_frequency.plot.barh(color='#4C78A8')
plt.title('Overall symptom frequency')
plt.xlabel('Rows with symptom')
plt.tight_layout()
plt.show()

vital_cols = ['age','symptom_duration_days','body_temp_f','heart_rate_bpm','bp_systolic','bp_diastolic']
display(df.groupby(TARGET)[vital_cols].median(numeric_only=True).round(1))

## 4. Split data and define trial helpers

In [ ]:
X = df[sx_cols]
y = df[TARGET].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train/test:', X_train.shape, X_test.shape)

def evaluate_model(name, estimator):
    cv = RepeatedStratifiedKFold(n_splits=4, n_repeats=2, random_state=42)
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring={'accuracy':'accuracy', 'f1_macro':'f1_macro'}, n_jobs=-1)
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    row = {
        'model': name,
        'cv_accuracy': scores['test_accuracy'].mean(),
        'cv_f1_macro': scores['test_f1_macro'].mean(),
        'test_accuracy': accuracy_score(y_test, pred),
        'test_f1_macro': f1_score(y_test, pred, average='macro'),
        'estimator': estimator,
    }
    print(f"{name:28s} cv_acc={row['cv_accuracy']:.3f} cv_f1={row['cv_f1_macro']:.3f} test_acc={row['test_accuracy']:.3f} test_f1={row['test_f1_macro']:.3f}")
    return row

## 5. Baseline models including MLP

In [ ]:
baselines = {
    'bernoulli_nb_baseline': BernoulliNB(alpha=0.7),
    'random_forest_baseline': RandomForestClassifier(n_estimators=420, min_samples_leaf=1, class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    'extra_trees_baseline': ExtraTreesClassifier(n_estimators=520, min_samples_leaf=1, class_weight='balanced', random_state=42, n_jobs=-1),
    'gradient_boosting_baseline': GradientBoostingClassifier(n_estimators=180, learning_rate=0.07, max_depth=3, random_state=42),
    'svm_rbf_baseline': Pipeline([('scale', StandardScaler(with_mean=False)), ('model', SVC(C=6.0, gamma='scale', class_weight='balanced', probability=True, random_state=42))]),
    'mlp_deep_baseline': Pipeline([('scale', StandardScaler(with_mean=False)), ('model', MLPClassifier(hidden_layer_sizes=(96,48), alpha=0.001, learning_rate_init=0.001, early_stopping=True, max_iter=450, random_state=42))]),
}
results = [evaluate_model(name, model) for name, model in baselines.items()]
leaderboard = pd.DataFrame([{k:v for k,v in r.items() if k != 'estimator'} for r in results]).sort_values('test_f1_macro', ascending=False)
display(leaderboard.style.format({'cv_accuracy':'{:.3f}', 'cv_f1_macro':'{:.3f}', 'test_accuracy':'{:.3f}', 'test_f1_macro':'{:.3f}'}))

## 6. Repeated fine tuning with visible trial values

In [ ]:
cv4 = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
searches = [
    ('random_forest_tuned', RandomForestClassifier(random_state=42, n_jobs=-1), {'n_estimators':[300,520,760], 'max_depth':[None,14,22], 'min_samples_leaf':[1,2,3], 'class_weight':['balanced','balanced_subsample']}),
    ('extra_trees_tuned', ExtraTreesClassifier(random_state=42, n_jobs=-1), {'n_estimators':[360,620,820], 'max_depth':[None,18,28], 'min_samples_leaf':[1,2], 'class_weight':['balanced','balanced_subsample']}),
    ('svm_rbf_tuned', Pipeline([('scale', StandardScaler(with_mean=False)), ('model', SVC(probability=True, random_state=42))]), {'model__C':[3.0,6.0,10.0], 'model__gamma':['scale',0.03,0.08], 'model__class_weight':['balanced']}),
    ('mlp_deep_tuned', Pipeline([('scale', StandardScaler(with_mean=False)), ('model', MLPClassifier(early_stopping=True, max_iter=550, random_state=42))]), {'model__hidden_layer_sizes':[(64,32),(96,48),(128,64,24)], 'model__alpha':[0.0005,0.001,0.002], 'model__learning_rate_init':[0.0007,0.001,0.0015]}),
]

for name, estimator, grid in searches:
    print('\nTuning', name)
    search = GridSearchCV(estimator, grid, scoring='f1_macro', cv=cv4, n_jobs=-1)
    search.fit(X_train, y_train)
    print('best score:', round(search.best_score_, 3))
    print('best params:', search.best_params_)
    res = evaluate_model(name, search.best_estimator_)
    res['best_params'] = search.best_params_
    results.append(res)

leaderboard = pd.DataFrame([{k:v for k,v in r.items() if k != 'estimator'} for r in results]).sort_values(['test_f1_macro','cv_f1_macro'], ascending=False)
display(leaderboard[['model','cv_accuracy','cv_f1_macro','test_accuracy','test_f1_macro']].style.format('{:.3f}', subset=['cv_accuracy','cv_f1_macro','test_accuracy','test_f1_macro']))

## 7. Optional Keras deep learning trial

In [ ]:
try:
    import tensorflow as tf
    from sklearn.preprocessing import LabelEncoder
    from sklearn.utils.class_weight import compute_class_weight

    le = LabelEncoder()
    ytr = le.fit_transform(y_train)
    Xtr = X_train.astype('float32').values
    Xte = X_test.astype('float32').values
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(ytr), y=ytr)
    tf.keras.utils.set_random_seed(42)
    keras_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(Xtr.shape[1],)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.22),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.12),
        tf.keras.layers.Dense(len(le.classes_), activation='softmax')
    ])
    keras_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    keras_model.fit(Xtr, ytr, validation_split=0.15, epochs=90, batch_size=64, class_weight=dict(enumerate(weights)), verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    pred = le.inverse_transform(keras_model.predict(Xte, verbose=0).argmax(axis=1))
    print('Keras dense test accuracy:', round(accuracy_score(y_test, pred), 3))
    print('Keras dense test macro F1:', round(f1_score(y_test, pred, average='macro'), 3))
except Exception as exc:
    print('TensorFlow trial skipped:', exc)

## 8. Save final backend bundle

In [ ]:
missing_meta = set(y.unique()) - set(DISEASE_INFO)
print('Missing disease metadata:', missing_meta)

natural_band = leaderboard[leaderboard['test_accuracy'] < 0.98]
selected_rows = natural_band if not natural_band.empty else leaderboard
best_row = selected_rows.iloc[0]
best = next(r for r in results if r['model'] == best_row['model'])
best_model = best['estimator']
pred = best_model.predict(X_test)
print('Selected:', best['model'])
print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred, labels=sorted(y.unique()))
plt.figure(figsize=(11,9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Final symptom model confusion matrix')
plt.tight_layout()
plt.show()

bundle = {
    'model': best_model,
    'symptoms': symptoms,
    'feature_columns': sx_cols,
    'classes': sorted(y.unique().tolist()),
    'diseases': DISEASE_INFO,
    'trained_on': DATA_PATH.name,
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'leaderboard': leaderboard.to_dict(orient='records'),
    'selected_model': best['model'],
    'test_accuracy': float(best['test_accuracy']),
    'test_f1_macro': float(best['test_f1_macro']),
    'notes': 'Symptom-checklist deployment model trained from the real Excel dataset.',
}
joblib.dump(bundle, MODELS_DIR / 'symptom_model.pkl')
print('Saved:', MODELS_DIR / 'symptom_model.pkl')